In [1]:
# 크롤링 모듈 임포트
from bs4 import BeautifulSoup 
from selenium import webdriver 

import time
import pandas as pd

In [2]:
# 스타벅스코리아 홈페이지 접속
driver = webdriver.Chrome('C:\\r_selenium\chromedriver')
url = 'https://www.starbucks.co.kr/store/store_map.do?disp=locale'
driver.get(url)

In [3]:
# 도시별 xpath값 저장
city_xpath = []
for i in range(17):
    text = f'//*[@id="container"]/div/form/fieldset/div/section/article[1]/article/article[2]/div[1]/div[2]/ul/li[{i+1}]/a'
    city_xpath.append(text)

In [4]:
#열려있는 페이지의 소스를 가져오기
html = driver.page_source
soup = BeautifulSoup(html,'html.parser')

In [5]:
# 도시별 이름 저장
city  = soup.select('ul.sido_arae_box')[0].text
city_name = []
leng = 2
for i in range(0,34,2):
    city_name_temp = city[i:leng]
    city_name.append(city_name_temp)
    leng+=2

In [11]:
#전체 버튼 정의
total_btn ='/html/body/div[4]/div[7]/div/form/fieldset/div/section/article[1]/article/article[2]/div[2]/div[3]/div/div/div[1]/ul/li[1]/a'

In [56]:
columns = ['매장명', '위도', '경도','매장타입','주소','전화번호']
startbucks_df_all = pd.DataFrame(columns = columns)  #strabuck전국매장 담을 df
st_cnt =1

for i in range(len(city_name)):

    time.sleep(3)
    # 해당 지역 클릭
    driver.find_element_by_xpath(city_xpath[i]).click()
    time.sleep(3)
    # 세종시 부분 처리를 위한 if문
    if(i<16):
        # 전체 버튼 클릭
        driver.find_element_by_xpath(total_btn).click()
        time.sleep(5)    
    # 스타벅스 서울시 매장 리스트 HTML 파싱
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    starbucks_soup_list = soup.select('li.quickResultLstCon')
    startbucks_list = []

    for item in starbucks_soup_list:
        name = item.select('strong')[0].text.strip() # 매장명
        lat = item['data-lat'].strip()
        long = item['data-long'].strip()
        # store_type = item.select('i')[0].text.strip() 
        store_type = item.select('i')[0].attrs.get('class')[0][4::]  #class이름 pin_다음이 매장의 type을 나타내는 아이콘임 
        adress = str(item.select('p')[0]).split('<br/>')[0].split('>')[1]
        tel = str(item.select('p')[0]).split('<br/>')[1].split('<')[0]
        startbucks_list.append([name,lat,long,store_type,adress,tel])
        st_cnt = st_cnt + 1 #매장총건수 카운트
    startbucks_df =pd.DataFrame(startbucks_list,columns=columns) #현재 지역 매장 담을 df
    # startbucks_df.to_excel(f'./startbucks_store_list_{i+1}_{city_name[i]}.xlsx', index = False)
    startbucks_df_all = startbucks_df_all.append(startbucks_df) #전국 df 결합 
    if(i<16):
        #지역 검색 클릭
        driver.find_element_by_xpath('//*[@id="container"]/div/form/fieldset/div/section/article[1]/article/header[2]/h3/a').click()

    print(f'{i+1}번째 {city_name[i]} 크롤링 종료')

print(f'크롤링을 종료합니다. 총매장수 : {st_cnt}')
startbucks_df_all.to_excel('./startbucks_store_list_20211031.xlsx', index = False)


1번째 서울 크롤링 종료
2번째 경기 크롤링 종료
3번째 광주 크롤링 종료
4번째 대구 크롤링 종료
5번째 대전 크롤링 종료
6번째 부산 크롤링 종료
7번째 울산 크롤링 종료
8번째 인천 크롤링 종료
9번째 강원 크롤링 종료
10번째 경남 크롤링 종료
11번째 경북 크롤링 종료
12번째 전남 크롤링 종료
13번째 전북 크롤링 종료
14번째 충남 크롤링 종료
15번째 충북 크롤링 종료
16번째 제주 크롤링 종료
17번째 세종 크롤링 종료
크롤링을 종료합니다. 총매장수 : 1610


In [57]:
startbucks_df_all

,매장명,위도,경도,매장타입,주소,전화번호
0,역삼아레나빌딩,37.501087,127.043069,general,서울특별시 강남구 언주로 425 (역삼동),1522-3232
1,논현역사거리,37.510178,127.022223,general,서울특별시 강남구 강남대로 538 (논현동),1522-3232
2,신사역성일빌딩,37.514132,127.020563,general,서울특별시 강남구 강남대로 584 (논현동),1522-3232
3,국기원사거리,37.499517,127.031495,general,서울특별시 강남구 테헤란로 125 (역삼동),1522-3232
4,스탈릿대치R,37.494668,127.062583,reserve,서울특별시 강남구 남부순환로 2947 (대치동),1522-3232
5,봉은사역,37.515000,127.063196,general,서울특별시 강남구 봉은사로 619 (삼성동),1522-3232
6,압구정윤성빌딩,37.5227934,127.0286009,general,서울특별시 강남구 논현로 834 (신사동),1522-3232
7,코엑스별마당,37.510150,127.060275,general,서울특별시 강남구 영동대로 513 (삼성동),1522-3232
8,삼성역섬유센터R,37.507750,127.060651,reserve,서울특별시 강남구 테헤란로 518 (대치동),1522-3232
9,압구정R,37.5273669,127.033061,reserve,서울특별시 강남구 언주로 861 (신사동),1522-3232
